# R 语言统计分析

使用 R 内置数据集进行假设检验、回归分析和置信区间计算。

无需下载数据或安装软件包 — 仅使用基础 R (Base R)。

## 1. 描述性统计

In [ ]:
data(mtcars)
cat("Dataset: mtcars (", nrow(mtcars), "cars, ", ncol(mtcars), "variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. 双样本 t 检验

手动挡汽车的燃油经济性 (MPG) 是否优于自动挡汽车？

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automatic:", round(mean(auto), 1), "mpg (n =", length(auto), ")\n")
cat("Manual:   ", round(mean(manual), 1), "mpg (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusion:",
    ifelse(t_result$p.value < 0.05,
           "Reject H0 — manual cars have significantly higher MPG",
           "Fail to reject H0"))

## 3. 卡方检验

气缸数与变速箱类型是否相互独立？

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automatic", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. 多元线性回归

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. 回归诊断

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. 置信区间

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% Confidence Intervals:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimate", ylab = "",
     main = "95% Confidence Intervals for Coefficients")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. 单因素方差分析 (One-Way ANOVA)

不同气缸数量之间的燃油效率 (MPG) 是否存在显著差异？

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nTukey HSD Post-hoc Comparisons:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG by Cylinder Count",
        xlab = "Cylinders", ylab = "Miles per Gallon",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## 总结

- **Welch t 检验**：在未调整的单尾比较中，手动挡汽车的平均 MPG 更高
- **卡方检验**：列联表提示存在关联，但较低的期望频数触发了近似警告，因此需谨慎解读该结果
- **回归分析**：在调整后，车重和马力 (horsepower) 是显著的负向预测因子；在该模型中变速箱类型不显著
- **方差分析 (ANOVA)**：4 缸、6 缸和 8 缸组之间的 MPG 差异显著；Tukey 检验结果指出了两两之间的具体差异